# SR-WM × AnyGoal — Google Colab 一键训练与评测模板

本 Notebook 用于在 Google Colab（T4 / L4 / A100）上复现与推进 **SR-WM（Sparse-Regime Relational World Model）** 在 `AnyGoal (arXiv:2610.26844)` 组合泛化基准上的训练与评测。

### 使用前准备
1. 在 Colab 菜单栏选择 **Runtime → Change runtime type → GPU**（推荐 L4 / A100，T4 也可跑状态空间实验）。
2. 将本地 `srwm_anygoal/` 推送到你的 GitHub 私有/公开仓库，或直接打包上传至 Google Drive 的 `/MyDrive/srwm_anygoal/`。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive（用于持久化数据集、权重与实验日志）
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_WORK = '/content/drive/MyDrive/srwm_anygoal_work'
os.makedirs(DRIVE_WORK, exist_ok=True)
os.environ['ANYGOAL_HOME'] = f'{DRIVE_WORK}/anygoal_home'
os.environ['STABLEWM_HOME'] = f'{DRIVE_WORK}/anygoal_swm'
os.environ['ANYGOAL_DATA_ROOT'] = f'{DRIVE_WORK}/anygoal_data'
os.environ['HF_HOME'] = '/content/hf_cache'  # 临时缓存放 Colab 本地 SSD，省 Drive 空间
os.environ['WANDB_MODE'] = 'disabled'
print('Workdir ready:', DRIVE_WORK)

In [ ]:
# 2. 一键安装本项目与 AnyGoal（使用独立 Python 3.11 环境，避开 Colab 默认 3.13 不兼容）
!rm -rf /content/srwm_anygoal
!git clone https://github.com/mordanours/srwm_anygoal.git /content/srwm_anygoal

!apt-get update -qq && apt-get install -y -qq swig
!pip install -q uv
!uv venv --clear /content/venv311 --python 3.11
!uv pip install --python /content/venv311/bin/python -e /content/srwm_anygoal "anygoal[eval]==0.1.0"

In [ ]:
# 3. 冒烟复现 G0 双解离实验（验证 Colab GPU 管线，约 1–2 分钟）
!/content/venv311/bin/python /content/srwm_anygoal/scripts/run_g0.py --steps 3000 --seeds 0,1,2 --out /content/drive/MyDrive/srwm_anygoal_work/g0_colab.json

In [ ]:
# 4. （G1b 阶段）在 Colab 临时盘下载官方 AnyGoal 数据集并抽成轻量纯状态 .npz 存回 Drive
# 仅保留 state/proprio/action，避免把 12.5GB 视频帧长期留在 Google Drive
from pathlib import Path
from huggingface_hub import hf_hub_download

tmp_dir = Path('/content/tmp_anygoal')
tmp_dir.mkdir(exist_ok=True)
data_root = Path(os.environ['ANYGOAL_DATA_ROOT'])

# 4a. 下载 3 个轻量官方测试集并抽纯状态
for fn in ['avoidt_test.h5', 'pusht_antigoal_test.h5', 'stackt_test.h5']:
    p = hf_hub_download(repo_id='HazelNam/anygoal', filename=fn, repo_type='dataset', local_dir=str(tmp_dir))
    out_npz = data_root / (Path(fn).stem + '_state.npz')
    !/content/venv311/bin/python /content/srwm_anygoal/scripts/extract_anygoal_states.py {p} {out_npz}

# 4b. 下载官方 PushT 专家训练集 (12.5GB .h5.zst) 到临时盘 -> 抽成纯状态 .npz 存 Drive -> 自动清理临时盘
zst_path = hf_hub_download(repo_id='quentinll/lewm-pusht', filename='pusht_expert_train.h5.zst', repo_type='dataset', local_dir=str(tmp_dir))
train_npz = data_root / 'pusht_expert_train_state.npz'
!/content/venv311/bin/python /content/srwm_anygoal/scripts/extract_anygoal_states.py {zst_path} {train_npz} --cleanup-tmp
!rm -rf /content/tmp_anygoal
print('All state-only datasets ready in Drive:', list(data_root.glob('*_state.npz')))

In [ ]:
# 5. 运行 G1b 官方 PushT 训练集 -> 官方 AvoidT / PushT-Antigoal / StackT 评测（含 CEM 规划）
!SDL_VIDEODRIVER=dummy HF_HUB_OFFLINE=1 WANDB_MODE=disabled /content/venv311/bin/python /content/srwm_anygoal/scripts/run_g1b_pusht.py \
  --train-file /content/drive/MyDrive/srwm_anygoal_work/anygoal_data/pusht_expert_train_state.npz \
  --test-dir /content/drive/MyDrive/srwm_anygoal_work/anygoal_data \
  --steps 5000 --seeds 0,1,2 --run-cem \
  --out /content/drive/MyDrive/srwm_anygoal_work/g1b_pusht_colab.json

In [ ]:
# 6. 运行 G1c：通用 LFS-WM + 防塌缩残差锚定门控 + 分层反捷径评测 + 三任务闭环 CEM 规划
!git -C /content/srwm_anygoal pull --ff-only
!SDL_VIDEODRIVER=dummy HF_HUB_OFFLINE=1 WANDB_MODE=disabled /content/venv311/bin/python /content/srwm_anygoal/scripts/run_g1c_pusht.py \
  --train-file /content/drive/MyDrive/srwm_anygoal_work/anygoal_data/pusht_expert_train_state.npz \
  --test-dir /content/drive/MyDrive/srwm_anygoal_work/anygoal_data \
  --steps 5000 --seeds 0,1,2 --run-cem --cem-eps 20 \
  --out /content/drive/MyDrive/srwm_anygoal_work/g1c_pusht_colab.json